# ES / MES 1-minute RTH data download

Run the cell below. Each year should print **bars/day median 390**, **first bar 09:31**, and **on 0.25 grid 100%**.

- Set `USE_MICRO = True` for MES.
- If a year hits a memory limit, set `YEARS` to one year at a time.
- CSVs are written to this project's file list; right-click to download.

In [ ]:
# ===== ES / MES 1-minute RTH download (QuantConnect Research) =====
from AlgorithmImports import *
import pandas as pd, numpy as np

USE_MICRO = False                  # True = MES, False = ES
YEARS     = [2022, 2023, 2024, 2025, 2026]

qb = QuantBook()
ticker = Futures.Indices.MICRO_SP_500_E_MINI if USE_MICRO else Futures.Indices.SP_500_E_MINI
fut = qb.add_future(ticker, Resolution.MINUTE,
                    extended_market_hours=True,
                    data_normalization_mode=DataNormalizationMode.RAW,      # real prices, 0.25 tick grid
                    data_mapping_mode=DataMappingMode.OPEN_INTEREST,        # most active contract
                    contract_depth_offset=0)
name = 'MES' if USE_MICRO else 'ES'

def clean(hist):
    df = hist.reset_index()
    df = df[['time', 'open', 'high', 'low', 'close', 'volume']].set_index('time').sort_index()
    df = df[~df.index.duplicated(keep='last')]               # roll overlaps
    if df.index.tz is None:
        df.index = df.index.tz_localize('America/New_York')
    return df

def fix_timezone(df):
    # The cash open shows up as a volume spike. In ET it lands on the 09:31 bar (bars are stamped at END time).
    vol_by_min = df.groupby(df.index.strftime('%H:%M'))['volume'].mean()
    spike = vol_by_min.loc['07:00':'11:00'].idxmax()
    if spike == '08:31':                                      # data is Chicago time -> convert
        df.index = df.index.tz_localize(None).tz_localize('America/Chicago').tz_convert('America/New_York')
        print('  converted Chicago -> New York')
    elif spike != '09:31':
        print(f'  WARNING: open volume spike at {spike}, expected 09:31 - check timestamps')
    return df

for year in YEARS:
    hist = qb.history(fut.symbol, datetime(year, 1, 1), datetime(year + 1, 1, 1), Resolution.MINUTE)
    if hist is None or len(hist) == 0:
        print(year, 'no data'); continue
    df = fix_timezone(clean(hist))
    df = df.between_time('09:31', '16:00')                    # regular trading hours only
    per_day = df.groupby(df.index.date).size()
    first_bar = pd.Series(df.index.strftime('%H:%M'), index=df.index).groupby(df.index.date).min().mode()[0]
    on_grid = ((df['close'] * 4).round(6) % 1 == 0).mean()
    print(f"{year}: {len(df):,} bars | {df.index.min()} -> {df.index.max()} | "
          f"bars/day median {per_day.median():.0f} | first bar {first_bar} | on 0.25 grid {on_grid:.0%}")
    df.to_csv(f'{name}_1m_RTH_{year}.csv')

print('done - files are in the project file list on the left; right-click to download')
